# Assignment 02: Reproducing SlowFast 4x16 R50 on THETIS RGB

Reproduction of Hovad, Hougaard-Jensen & Clemmensen (2024), *Classification of Tennis Actions Using Deep Learning*, arXiv:2402.02545, DOI 10.48550/arXiv.2402.02545.

**Run on Kaggle:** Settings → Accelerator **GPU T4 x1 or P100**, Internet **on**. Add the THETIS RGB videos as an input dataset (see README). Then *Save Version → Save & Run All (Commit)* so the ~5 h run survives closing the tab. If a session ends early, run the notebook again with the previous output attached: training resumes from `last.pt`.

In [ ]:
# 1. Get the code and dependencies
import os, sys
REPO_URL = "https://github.com/<your-username>/slowfast-thetis-repro.git"   # <-- edit
if not os.path.exists("slowfast-thetis-repro"):
    !git clone -q $REPO_URL
%cd slowfast-thetis-repro
!pip install -q pytorchvideo==0.1.5 fvcore
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Find the THETIS RGB videos (all of /kaggle/input is searched; depth/mask/skeleton folders are skipped)
import glob
n = len(glob.glob('/kaggle/input/**/*.avi', recursive=True))
print('AVI files found under /kaggle/input:', n)
assert n > 0, 'No videos found: attach your THETIS dataset with Add Input'
CFG = "configs/slowfast_4x16_r50_thetis.yaml"
OPTS = "data.root=/kaggle/input"


In [ ]:
# 3. Dataset: index videos, cache frames (short side 256), 70/20/10 stratified split, figures
!python src/prepare_thetis.py --config $CFG --opts $OPTS

In [ ]:
# 4. Model summary (stage shapes, parameters, GFLOPs)
!python src/model.py --config $CFG --out /kaggle/working/runs/slowfast_4x16_r50/model_summary.json

In [ ]:
# 5. Train from scratch: SGD + Nesterov, cosine LR with 34-epoch warmup, 196 epochs, best-val checkpoint
!python src/train.py --config $CFG --opts $OPTS --time-limit-hours 11

In [ ]:
# 6. Test with the paper's protocol (2 clips x 3 crops) and build the comparison table
!python src/evaluate.py --config $CFG --opts $OPTS

In [ ]:
# 7. Show results
from IPython.display import Image, Markdown, display
import json
RUN = "/kaggle/working/runs/slowfast_4x16_r50"
display(Markdown(open(f"{RUN}/comparison_table.md").read()))
print(json.dumps(json.load(open(f"{RUN}/results.json")), indent=1))
for f in ["training_curves", "confusion_matrix", "per_class_accuracy", "class_distribution", "sample_frames"]:
    display(Image(f"{RUN}/figures/{f}.png", width=900))

In [ ]:
# 8. Bundle everything the report and repo need (checkpoint kept separately: it is ~135 MB)
!cd /kaggle/working && zip -qr results_bundle.zip runs/slowfast_4x16_r50 -x '*last.pt' -x '*best.pt' && ls -lh results_bundle.zip runs/slowfast_4x16_r50/*.pt